# Reports as artifacts

Reports can be stored with a dataset, shared in an issue, passed to an
agent, or compared across runs. This chapter covers JSON, Markdown, the
context-budgeted digest, the Python dict, and the conditions under which
serialization is deterministic.
The field-level reference is [](../appendix/report_contract.ipynb).

The fixture contains a region-code mapping and a column equal to twice the quantity:

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(4)
n = 800

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["red", "green", "blue"][r] for r in region_id]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]

df = pl.DataFrame({
    "order_id": list(range(n)),
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
})

report = onceover.profile(df)
print(report.overview["headline"])

## JSON

`to_json()` returns the complete report — overview, columns, findings,
groups, selection summary, coverage — as one deterministic, schema-versioned
string. `report.data` parses it into a dict; `save(path)` writes it;
`json.dumps(report.data, indent=2)` formats it for reading.
`schema_version` (currently `"0.2"`) identifies the report format. Changes
are usually additive, but the 0.1 → 0.2 revision changed the finding
taxonomy incompatibly. Readers should check the schema version and
tolerate unknown fields and categories.

## Determinism

With the same package version, input, and settings, repeated runs produce
identical JSON when no deadline is set and timing is disabled. A deadline can change which passes
finish. Coverage records those skips.

Two different tables can produce the same summary, so matching report JSON
does not prove that the input data is identical.

In [ ]:
first = onceover.profile(df)
second = onceover.profile(df, budget="standard")

print("byte-identical:", first.to_json() == second.to_json())
print("schema_version:", report.data["schema_version"])
print("selection:", report.data["selection"])

`selection.candidates` counts the detectors' candidates before consolidation.
`selection.distinct_statements` counts the different claims after matching
candidates are combined; `selection.selected` counts the findings retained
under the selection limits. Compare these counts when a report contains
fewer findings than the detectors produced.

## Markdown

`to_markdown()` renders the blocks as a compact document — headline, ranked
findings with their basis and form, the related-finding groups, a column
table, and the coverage table. It is the format meant for pasting into a
pull request or handing to a person. In a notebook, the report object
renders itself this way automatically:

In [ ]:
print(report.to_markdown())

## Digest

`report.to_digest(budget_chars=4_000)` renders a plain-text digest sized to a
reading budget — built for the case where the report's JSON is larger
than the context it has to fit into. The digest has a header (shape,
schema version, headline, coverage exceptions, finding counts by
category), then one line per rollup group: findings that share
category, type, form, and target, carrying the group's strongest
claim, strongest first, with a `(xN)` marker when a group rolls up N
findings. When the groups exceed the budget, the digest keeps as many
as fit and ends by stating how many were omitted; the result never
exceeds the budget. At very small budgets the header itself degrades
first — category counts, then the coverage line — before anything is
truncated.

A digest is an index into the report, not a summary of record. A
rollup line names a group and its strongest claim; the group's other
members, and every finding's evidence, remain in the JSON. Two
findings that share a rollup key can differ in columns and conditions,
so the marker is a prompt to look, not a claim that the members are
interchangeable.

In [ ]:
print(report.to_digest(500))

## Feeding an agent

The function below copies each finding's `claim` verbatim and includes its
id and category. It summarizes the evidence record with the largest
`rows_evaluated`, giving its basis, validation method, and row count.
The complete evidence remains available in the report.

In [ ]:
def brief(f):
    primary = max(f["evidence"], key=lambda e: e["scope"]["rows_evaluated"] or 0)
    return {
        "id": f["id"],
        "category": f["category"],
        "type": f.get("type"),
        "claim": f["claim"],
        "basis": primary["basis"],
        "validation": primary["validation"]["method"],
        "rows_evaluated": primary["scope"]["rows_evaluated"],
        "interpretation": f.get("interpretation"),
    }

digest = {
    "headline": report.overview["headline"],
    "top_findings": [brief(f) for f in report.findings[:4]],
    "coverage": report.coverage,
}
print(json.dumps(digest["top_findings"][0], indent=2))

Keep the basis, row count, and validation method with the claim when
sharing this digest. The record with the most evaluated rows is not
necessarily the best validated: inspect the full evidence before choosing
a result for an automated action. When no finding is reported, check coverage to see whether the detector ran and
what rows and candidates it examined.

## The Rust API

The same profiler is a Rust crate over Polars 0.55, for pipelines that
never leave Rust:

```toml
[dependencies]
onceover = "0.1"
polars = "0.55"
```

```rust
use onceover::{profile, Budget, ProfileConfig};
use polars::prelude::*;

fn main() -> PolarsResult<()> {
    let df = df!("id" => [1i64, 2, 3], "value" => [4.0, 5.0, 6.0])?;

    let cfg = ProfileConfig {
        budget: Budget::Deep,
        ..ProfileConfig::default()
    };
    let report = profile(&df, &cfg);

    println!("{}", report.overview.headline);
    println!("{}", report.to_json());   // same schema, same determinism
    Ok(())
}
```

`ProfileConfig::default()` uses the `standard` budget, default detectors,
the default selection policy, and no deadline; the `selection` field is the
same `SelectionPolicy` that caps findings and rules in Python via
`max_findings`/`max_rules`. The report schema is identical across
languages, so a Rust service and a Python notebook produce comparable
artifacts from the same table.

## Compare profiles

`before.diff(after)` compares statements by stable ID and records added,
removed, and changed claims or scores, plus coverage-status changes. It is
useful after a new extract arrives or a transformation changes the table.
Evidence-only changes with unchanged claim and score are not counted as
changes. This comparison is distinct from the model-routing drift pass.

The example appends duplicate rows to the orders fixture:


In [ ]:
after = onceover.profile(pl.concat([df, df.head(3)]))
comparison = report.diff(after)
print("Added findings:")
for finding in comparison["added"]:
    print(finding["category"], finding["claim"])
print("Removed:", len(comparison["removed"]), "Changed:", len(comparison["changed"]))
print("Coverage status changes:", comparison["coverage_changes"])


## Budgeted JSON

`to_json(max_chars=...)` retains high-ranked findings in full and indexes
others. If needed it reduces column-card and coverage detail. Its `budget`
record states what was omitted; it raises `ValueError` if the minimal
index cannot fit. Use `to_json()` without a size limit to retain the whole
artifact. This is a presentation budget, separate from discovery budgets.

The [API reference](api.ipynb) lists the report methods and parameters.
